In [1]:
# %% [markdown]
# # 05a_exposure_final_check.ipynb
# Final exposure verification at full scale: confirms 0/15 sensitive
# columns across the ACTUAL 60 questions used in the full experiment
# (not just a handful of hand-picked samples), by rebuilding the exact
# v2 prompt for every question and evidence pair and scanning each one.
#
# This is the number that matters most for the manuscript's RQ2/H2
# claim, so it gets checked exhaustively rather than spot-checked.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import pandas as pd
from utils import build_v2_prompt, measure_exposure, SENSITIVE_COLUMNS, SENSITIVE_CODES

with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    questions = json.load(f)

print(f"Checking exposure across all {len(questions)} questions' actual prompts...")

# %% [markdown]
# ## 1. Exhaustive per-question exposure scan

# %%
results = []
for q in questions:
    prompt = build_v2_prompt(q['question'], q['evidence'])
    exp = measure_exposure(prompt)
    results.append({
        'question_id': q['question_id'],
        'col_exposed': exp['col_exposed'],
        'col_total': exp['col_total'],
        'code_exposed': exp['code_exposed'],
        'code_total': exp['code_total'],
        'prompt_chars': len(prompt),
    })

df_exp = pd.DataFrame(results)

print(f"\nMax columns exposed across all {len(df_exp)} questions: {df_exp['col_exposed'].max()} / {len(SENSITIVE_COLUMNS)}")
print(f"Max codes exposed across all {len(df_exp)} questions: {df_exp['code_exposed'].max()} / {len(SENSITIVE_CODES)}")
print(f"Prompt length range: {df_exp['prompt_chars'].min():,} - {df_exp['prompt_chars'].max():,} chars")
print(f"Prompt length mean: {df_exp['prompt_chars'].mean():,.0f} chars")

any_leak = df_exp[(df_exp['col_exposed'] > 0) | (df_exp['code_exposed'] > 0)]
if len(any_leak) > 0:
    print(f"\n!! LEAK DETECTED in {len(any_leak)} question(s):")
    print(any_leak)
else:
    print(f"\n✓ CONFIRMED: 0/{len(SENSITIVE_COLUMNS)} sensitive columns and "
          f"0/{len(SENSITIVE_CODES)} internal codes exposed across all "
          f"{len(df_exp)} questions' actual v2 prompts.")

# %% [markdown]
# ## 2. Cross-check against v1 for the same 60 questions (same-basis
#     comparison for the manuscript's Table 3 replacement)

# %%
from utils import build_sl_prompt_schema, FINANCIAL_TABLES, FEW_SHOT_EXAMPLES

def build_v1_full_prompt(question, evidence):
    """Mirrors query_semantic_layer()'s actual prompt construction,
    without making the API call, for exposure measurement only."""
    sl_schema = build_sl_prompt_schema()
    table_ref = "\n".join(
        f"  {t}: {', '.join(c)}" for t, c in FINANCIAL_TABLES.items()
    )
    return f"""You are an expert SQLite query generator using a Semantic Layer.

## Semantic Layer (Concept → Actual SQL Mapping)
{sl_schema}

## Exact Table and Column Names (use ONLY these)
{table_ref}

## Critical Rules
1. Concept names are NOT column names. Always translate:
   - 'region'          → A3   (district table)
   - 'district_name'   → A2   (district table)
   - 'avg_salary'      → A11  (district table)
   - 'frequency_label' → use CASE frequency WHEN 'POPLATEK MESICNE' THEN 'monthly' ... END
   - 'status_label'    → use CASE status WHEN 'A' THEN 'completed_ok' ... END
2. Always prefix every column with its table name.
3. Exact table names: account, client, disp, loan, trans, card, order, district
   NOTE: use 'order' not 'orders'.
4. SQL clause order: SELECT → FROM → JOIN → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT.
5. "eligible for loans" = has a record in the loan table. Do NOT filter by loan.status.
6. Use standard SQLite functions only (IFNULL not ISNULL, no NUMBER()).
7. Return ONLY valid SQLite SQL. No explanation, no markdown, no backticks.
{FEW_SHOT_EXAMPLES}

## Question
{question}

## Additional Evidence
{evidence if evidence else 'None'}
"""

v1_results = []
for q in questions[:5]:  # sample check is enough for v1 since it's already confirmed structurally fixed at 15/15
    prompt = build_v1_full_prompt(q['question'], q['evidence'])
    exp = measure_exposure(prompt)
    v1_results.append({'question_id': q['question_id'], 'col_exposed': exp['col_exposed']})

df_v1 = pd.DataFrame(v1_results)
print("v1 exposure (sample of 5, for cross-check against the already-confirmed 15/15 finding):")
print(df_v1)

# %% [markdown]
# ## 3. Summary table for the manuscript

# %%
print("\n" + "=" * 70)
print("CORRECTED Table 3 replacement (schema-level exposure)")
print("=" * 70)
print(f"""
  Metric                          Text-to-SQL   Semantic Layer v1   Semantic Layer v2
  Sensitive columns exposed              15              15  (0%)         0  (100%)
  Internal codes exposed                  0               0                0
  Schema context size (chars)           ~976           ~3,555          {df_exp['prompt_chars'].mean():>6,.0f} (mean)

  v1's originally-claimed 100% reduction is NOT supported by the
  actual v1 prompt (see prior exposure recheck) -- v1 exposes all 15
  columns via its concept-to-SQL mapping table, table reference block,
  and few-shot examples.

  v2 (the corrected true-information-hiding architecture) achieves the
  100% reduction v1 originally claimed, verified exhaustively across
  all 60 questions' actual prompts, not just a schema-only snippet.
""")

SCRIPT VERSION: 2026-07-15-v1
Checking exposure across all 60 questions' actual prompts...

Max columns exposed across all 60 questions: 0 / 15
Max codes exposed across all 60 questions: 0 / 9
Prompt length range: 14,631 - 14,850 chars
Prompt length mean: 14,708 chars

✓ CONFIRMED: 0/15 sensitive columns and 0/9 internal codes exposed across all 60 questions' actual v2 prompts.
v1 exposure (sample of 5, for cross-check against the already-confirmed 15/15 finding):
  question_id  col_exposed
0        CR01           15
1        CR02           15
2        CR03           15
3        CR04           15
4        CR05           15

CORRECTED Table 3 replacement (schema-level exposure)

  Metric                          Text-to-SQL   Semantic Layer v1   Semantic Layer v2
  Sensitive columns exposed              15              15  (0%)         0  (100%)
  Internal codes exposed                  0               0                0
  Schema context size (chars)           ~976           ~3,555     